In [2]:
import math
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPolygon

import folium
from folium import Choropleth, Marker
from folium.plugins import HeatMap, MarkerCluster

In [16]:
collisions = gpd.read_file("NYPD_Motor_Vehicle_Collisions.shp")
hospitals = gpd.read_file("nyu_2451_34494.shp")
hospitals.head()

,id,name,address,zip,factype,facname,capacity,capname,bcode,xcoord,ycoord,latitude,longitude,geometry
0,317000001H1178,BRONX-LEBANON HOSPITAL CENTER - CONCOURSE DIVI...,1650 Grand Concourse,10457,3102,Hospital,415,Beds,36005,1008872.0,246596.0,40.843490,-73.911010,POINT (1008872 246596)
1,317000001H1164,BRONX-LEBANON HOSPITAL CENTER - FULTON DIVISION,1276 Fulton Ave,10456,3102,Hospital,164,Beds,36005,1011044.0,242204.0,40.831429,-73.903178,POINT (1011044 242204)
2,317000011H1175,CALVARY HOSPITAL INC,1740-70 Eastchester Rd,10461,3102,Hospital,225,Beds,36005,1027505.0,248287.0,40.848060,-73.843656,POINT (1027505 248287)
3,317000002H1165,JACOBI MEDICAL CENTER,1400 Pelham Pkwy,10461,3102,Hospital,457,Beds,36005,1027042.0,251065.0,40.855687,-73.845311,POINT (1027042 251065)
4,317000008H1172,LINCOLN MEDICAL & MENTAL HEALTH CENTER,234 E 149 St,10451,3102,Hospital,362,Beds,36005,1005154.0,236853.0,40.816758,-73.924478,POINT (1005154 236853)


In [12]:
m_1 = folium.Map(location=[40.7, -74], zoom_start=11) 
HeatMap(data=collisions[['LATITUDE','LONGITUDE']], radius=10).add_to(m_1)
m_1

In [32]:
m_2 = folium.Map(location=[40.7, -74], zoom_start=11) 

for index, row in hospitals.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=row['name'],
        icon=folium.Icon(color='red', icon='hospital', prefix='fa')).add_to(m_2)
# Add buffer 10km around 
buffer = hospitals.geometry.buffer(10000)
folium.GeoJson(buffer.to_crs(epsg=2263)).add_to(m_2)
m_2

In [39]:
outside_range_list = []

for i, row in collisions.iterrows():
    if union.contains(collisions.iloc[i].geometry) == False:
        outside_range_list.append(row)
        
outside_range = pd.DataFrame(outside_range_list)

In [40]:
percentage = round(100*len(outside_range)/len(collisions), 2)
print("Percentage of collisions more than 10 km away from the closest hospital: {}%".format(percentage))

Percentage of collisions more than 10 km away from the closest hospital: 15.12%


In [41]:
def best_hospital(collision_location):
    # Your code here
    distance = hospitals.geometry.distance(collision_location)
    name = hospitals.iloc[distance.idxmin()]['name']
    return name

# Test your function: this should suggest CALVARY HOSPITAL INC
print(best_hospital(outside_range.geometry.iloc[0]))

CALVARY HOSPITAL INC


In [45]:
highest_demand = outside_range.geometry.apply(best_hospital).value_counts().idxmax()